Task 3: Machine Learning Model Comparison on HIGGS Dataset

In [1]:
#Import required libraries for pyspark
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, when

In [2]:
# Initialising spark session
spark = SparkSession.builder \
    .appName("FraudDetectionTask3") \
    .master("local[2]") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()
print("Spark session created successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 18:58:58 WARN Utils: Your hostname, Sais-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.223.134.171 instead (on interface en0)
26/06/26 18:58:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 18:58:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 18:58:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/26 18:58:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark session created successfully


In [3]:
# Load processed data
processed_df = spark.read.parquet("task2_processed_data")

In [4]:
# Load pipeline model
pipeline_model = PipelineModel.load("task2_pipeline")

In [5]:
# Check whether processed data is loaded correctly
processed_df.printSchema()
processed_df.show(5)

root
 |-- label: double (nullable = true)
 |-- c1: double (nullable = true)
 |-- c2: double (nullable = true)
 |-- c3: double (nullable = true)
 |-- c4: double (nullable = true)
 |-- c5: double (nullable = true)
 |-- c6: double (nullable = true)
 |-- c7: double (nullable = true)
 |-- c8: double (nullable = true)
 |-- c9: double (nullable = true)
 |-- c10: double (nullable = true)
 |-- c11: double (nullable = true)
 |-- c12: double (nullable = true)
 |-- c13: double (nullable = true)
 |-- c14: double (nullable = true)
 |-- c15: double (nullable = true)
 |-- c16: double (nullable = true)
 |-- c17: double (nullable = true)
 |-- c18: double (nullable = true)
 |-- c19: double (nullable = true)
 |-- c20: double (nullable = true)
 |-- c21: double (nullable = true)
 |-- c22: double (nullable = true)
 |-- c23: double (nullable = true)
 |-- c24: double (nullable = true)
 |-- c25: double (nullable = true)
 |-- c26: double (nullable = true)
 |-- c27: double (nullable = true)
 |-- c28: double (null

26/06/26 19:12:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+------------------+-------------------+-------------------+------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+--------------------+
|label|                c1|                 c2|                 c3|                c4|                 c5|                c6|                  c7|                 c8|                c9|               c10|                c11|                c12|               c13|               c14|                 c15|                c16|              c17|                c18|                 c19|                 c20|  

Import required machine learning Libraries

In [6]:
import time

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.tuning import (
    ParamGridBuilder,
    CrossValidator
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

In [7]:
# Cache the processed data to speed up repeated model training
processed_df = processed_df.cache() 

 # Trigger caching by performing an action
processed_df.count() 

26/06/26 19:13:01 WARN MemoryStore: Not enough space to cache rdd_31_15 in memory! (computed 68.1 MiB so far)
26/06/26 19:13:01 WARN MemoryStore: Not enough space to cache rdd_31_14 in memory! (computed 132.1 MiB so far)
26/06/26 19:13:01 WARN BlockManager: Persisting block rdd_31_15 to disk instead.
26/06/26 19:13:01 WARN BlockManager: Persisting block rdd_31_14 to disk instead.
26/06/26 19:13:03 WARN MemoryStore: Not enough space to cache rdd_31_15 in memory! (computed 4.0 MiB so far)
26/06/26 19:13:03 WARN MemoryStore: Not enough space to cache rdd_31_14 in memory! (computed 236.0 MiB so far)
26/06/26 19:13:04 WARN MemoryStore: Not enough space to cache rdd_31_17 in memory! (computed 68.1 MiB so far)
26/06/26 19:13:04 WARN BlockManager: Persisting block rdd_31_17 to disk instead.
26/06/26 19:13:04 WARN MemoryStore: Not enough space to cache rdd_31_16 in memory! (computed 132.1 MiB so far)
26/06/26 19:13:04 WARN BlockManager: Persisting block rdd_31_16 to disk instead.
26/06/26 19:13

11000000

Train-Test Split

In [8]:
# Split dataset into 70% training and 30% testing
train_data, test_data = processed_df.randomSplit([0.7, 0.3], seed=42)

Evaluation Metrics

In [9]:
# AUC - measures how well the model separates classes
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# Accuracy - overall correctness of the prediction
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# F1 Score - shows how well the model balances correct predictions and missed cases
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# Precision – shows how many of the predicted positive cases are actually correct
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

Logistic Regression 

In [10]:
# Logistic regression model
lr = LogisticRegression(
    featuresCol="scaledfeatures",
    labelCol="label",
    maxIter=30
)

#Hyperparameter tuning
lr_paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.maxIter, [20, 50])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=auc_evaluator,
    numFolds=2
)

# Training the model + Measuring time
start = time.time()
lr_model = lr_cv.fit(train_data)
lr_time = time.time() - start

# Predictions
lr_predictions = lr_model.transform(test_data)

#  Evaluation Metrics Calculation
lr_auc = auc_evaluator.evaluate(lr_predictions)
lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)
lr_f1 = f1_evaluator.evaluate(lr_predictions)
lr_precision = precision_evaluator.evaluate(lr_predictions)

# Save the trained model
lr_model.bestModel.write().overwrite().save("lr_model")

print("Model trained and saved successfully")

26/06/26 19:15:15 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/26 19:23:06 WARN MemoryStore: Not enough space to cache rdd_31_3 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:08 WARN MemoryStore: Not enough space to cache rdd_31_7 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:12 WARN MemoryStore: Not enough space to cache rdd_31_13 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:44 WARN MemoryStore: Not enough space to cache rdd_31_3 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:46 WARN MemoryStore: Not enough space to cache rdd_31_7 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:48 WARN MemoryStore: Not enough space to cache rdd_31_11 in memory! (computed 236.0 MiB so far)
26/06/26 19:23:50 WARN MemoryStore: Not enough space to cache rdd_31_15 in memory! (computed 236.1 MiB so far)
                                                                                

Model trained and saved successfully


Logistic Regression Results Output

In [11]:
print("Logistic Regression Results")
print(f"Training Time: {lr_time:.2f} seconds")
print(f"AUC Score: {lr_auc:.4f}")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1 Score: {lr_f1:.4f}")
print(f"Precision: {lr_precision:.4f}")

# Displays confusion matrix to show correct and incorrect predictions for each class
lr_predictions.groupBy("label", "prediction").count().show()

Logistic Regression Results
Training Time: 481.33 seconds
AUC Score: 0.6790
Accuracy: 0.6352
F1 Score: 0.6298
Precision: 0.6359


[Stage 738:=====================================================> (35 + 1) / 36]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  1.0|       1.0|1301484|
|  0.0|       1.0| 756512|
|  1.0|       0.0| 448118|
|  0.0|       0.0| 795825|
+-----+----------+-------+



Decision Tree Classifier

In [12]:
# Decision tree classifier model
dt = DecisionTreeClassifier(
    featuresCol="scaledfeatures",
    labelCol="label",
    maxDepth=10,         
    maxBins=32        
)

#Hyperparameter grid
dt_paramGrid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10])   
    .build()
)

dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_paramGrid,
    evaluator=auc_evaluator,
    numFolds=2   
)

# Training the model + Measuring time
start = time.time()
dt_model = dt_cv.fit(train_data)
dt_time = time.time() - start

# Predictions
dt_predictions = dt_model.transform(test_data)

#  Evaluation Metrics 
dt_auc = auc_evaluator.evaluate(dt_predictions)
dt_accuracy = accuracy_evaluator.evaluate(dt_predictions)
dt_f1 = f1_evaluator.evaluate(dt_predictions)
dt_precision = precision_evaluator.evaluate(dt_predictions)

# Save the trained model
dt_model.bestModel.write().overwrite().save("dt_model")

print("Model trained and saved successfully")

26/06/26 19:28:52 WARN MemoryStore: Not enough space to cache rdd_31_1 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:53 WARN MemoryStore: Not enough space to cache rdd_31_3 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:54 WARN MemoryStore: Not enough space to cache rdd_31_5 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:55 WARN MemoryStore: Not enough space to cache rdd_31_7 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:56 WARN MemoryStore: Not enough space to cache rdd_31_9 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:57 WARN MemoryStore: Not enough space to cache rdd_31_11 in memory! (computed 68.1 MiB so far)
26/06/26 19:28:58 WARN MemoryStore: Not enough space to cache rdd_31_13 in memory! (computed 68.1 MiB so far)
26/06/26 19:29:00 WARN MemoryStore: Not enough space to cache rdd_31_15 in memory! (computed 68.1 MiB so far)
26/06/26 19:29:02 WARN MemoryStore: Not enough space to cache rdd_31_19 in memory! (computed 132.1 MiB so far)
26/06/26 19:29

Model trained and saved successfully


Decision Tree Classifier Results Output

In [15]:
print("Decision Tree Classifier Results")
print(f"Training Time: {dt_time:.2f} seconds")
print(f"AUC Score: {dt_auc:.4f}")
print(f"Accuracy: {dt_accuracy:.4f}")
print(f"F1 Score: {dt_f1:.4f}")
print(f"Precision: {dt_precision:.4f}")

# Display confusion matrix
dt_predictions.groupBy("label", "prediction").count().show()

Decision Tree Classifier Results
Training Time: 306.17 seconds
AUC Score: 0.6699
Accuracy: 0.7047
F1 Score: 0.7047
Precision: 0.7047


[Stage 1225:====================================================> (35 + 1) / 36]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  1.0|       1.0|1262085|
|  0.0|       1.0| 487536|
|  1.0|       0.0| 487517|
|  0.0|       0.0|1064801|
+-----+----------+-------+



Random Forest Classifier

In [16]:
# Random forest classifier model
rf = RandomForestClassifier(
    featuresCol="scaledfeatures",
    labelCol="label",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Hyperparameter grid
rf_paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 50])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_paramGrid,
    evaluator=auc_evaluator,
    numFolds=2 
)

# Training the model + Measuring time
start = time.time()
rf_model = rf_cv.fit(train_data)
rf_time = time.time() - start

# Predictions
rf_predictions = rf_model.transform(test_data)

#  Evaluation Metrics 
rf_auc = auc_evaluator.evaluate(rf_predictions)
rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)
rf_precision = precision_evaluator.evaluate(rf_predictions)

# Save the trained model
rf_model.bestModel.write().overwrite().save("rf_model")

print("Model trained and saved successfully")

26/06/26 20:12:45 WARN DAGScheduler: Broadcasting large task binary with size 1281.5 KiB
26/06/26 20:18:10 WARN DAGScheduler: Broadcasting large task binary with size 1071.0 KiB
26/06/26 20:18:51 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/06/26 20:19:45 WARN DAGScheduler: Broadcasting large task binary with size 1242.6 KiB
26/06/26 20:24:18 WARN DAGScheduler: Broadcasting large task binary with size 1281.7 KiB
26/06/26 20:29:47 WARN DAGScheduler: Broadcasting large task binary with size 1071.3 KiB
26/06/26 20:30:30 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/06/26 20:31:24 WARN DAGScheduler: Broadcasting large task binary with size 1236.3 KiB
26/06/26 20:32:15 WARN MemoryStore: Not enough space to cache rdd_31_1 in memory! (computed 68.1 MiB so far)
26/06/26 20:32:23 WARN MemoryStore: Not enough space to cache rdd_31_13 in memory! (computed 132.1 MiB so far)
26/06/26 20:32:24 WARN MemoryStore: Not enough space to cache rdd_31_15 in

Model trained and saved successfully


26/06/26 20:42:51 WARN TaskSetManager: Stage 1522 contains a task of very large size (1317 KiB). The maximum recommended task size is 1000 KiB.


Random Forest Classifier Results Output

In [17]:
print("Random Forest Results")
print(f"Training Time: {rf_time:.2f} seconds")
print(f"AUC Score: {rf_auc:.4f}")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1 Score: {rf_f1:.4f}")
print(f"Precision: {rf_precision:.4f}")

# Display confusion matrix
rf_predictions.groupBy("label", "prediction").count().show()

Random Forest Results
Training Time: 1905.04 seconds
AUC Score: 0.7696
Accuracy: 0.6974
F1 Score: 0.6965
Precision: 0.6971


26/06/26 20:45:44 WARN DAGScheduler: Broadcasting large task binary with size 1186.1 KiB
[Stage 1525:====================================================> (35 + 1) / 36]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  1.0|       1.0|1305002|
|  0.0|       1.0| 554508|
|  1.0|       0.0| 444600|
|  0.0|       0.0| 997829|
+-----+----------+-------+



26/06/26 20:46:16 WARN DAGScheduler: Broadcasting large task binary with size 1142.0 KiB
                                                                                

GBT Classifer

In [18]:
# GBT Classfier model
gbt = GBTClassifier(
    featuresCol="scaledfeatures",
    labelCol="label",
    maxIter=50,        
    maxDepth=5,        
    seed=42
)

# Hyperparameter grid
gbt_paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [30, 50])
    .build()
)

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_paramGrid,
    evaluator=auc_evaluator,
    numFolds=2  
)

# Training the model + Measuring time
start = time.time()
gbt_model = gbt_cv.fit(train_data)
gbt_time = time.time() - start

# Predictions
gbt_predictions = gbt_model.transform(test_data)

#  Evaluation Metrics
gbt_auc = auc_evaluator.evaluate(gbt_predictions)
gbt_accuracy = accuracy_evaluator.evaluate(gbt_predictions)
gbt_f1 = f1_evaluator.evaluate(gbt_predictions)
gbt_precision = precision_evaluator.evaluate(gbt_predictions)

# Save the trained model
gbt_model.bestModel.write().overwrite().save("gbt_model")

print("Model trained and saved successfully")

26/06/26 21:09:13 WARN MemoryStore: Not enough space to cache rdd_31_13 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:14 WARN MemoryStore: Not enough space to cache rdd_31_15 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:16 WARN MemoryStore: Not enough space to cache rdd_31_17 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:17 WARN MemoryStore: Not enough space to cache rdd_31_19 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:18 WARN MemoryStore: Not enough space to cache rdd_31_21 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:19 WARN MemoryStore: Not enough space to cache rdd_31_23 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:20 WARN MemoryStore: Not enough space to cache rdd_31_25 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:21 WARN MemoryStore: Not enough space to cache rdd_31_27 in memory! (computed 132.1 MiB so far)
26/06/26 21:09:22 WARN MemoryStore: Not enough space to cache rdd_31_29 in memory! (computed 132.1 MiB so far)
2

Model trained and saved successfully


GBT Classifier Results Outputs

In [19]:
print("GBT Classifier Results")
print(f"Training Time: {gbt_time:.2f} seconds")
print(f"AUC Score: {gbt_auc:.4f}")
print(f"Accuracy: {gbt_accuracy:.4f}")
print(f"F1 Score: {gbt_f1:.4f}")
print(f"Precision: {gbt_precision:.4f}")

# Display confusion matrix
gbt_predictions.groupBy("label", "prediction").count().show()

GBT Classifier Results
Training Time: 1757.12 seconds
AUC Score: 0.7953
Accuracy: 0.7182
F1 Score: 0.7181
Precision: 0.7180


[Stage 4736:====================================================> (35 + 1) / 36]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  1.0|       1.0|1294156|
|  0.0|       1.0| 475055|
|  1.0|       0.0| 455446|
|  0.0|       0.0|1077282|
+-----+----------+-------+



Model Performance Comparison Table

In [20]:
import pandas as pd

# Create a list containing evaluation metrics results of all 4 ML models
data = [
    ("Logistic Regression", lr_accuracy, lr_auc, lr_f1, lr_precision, lr_time),
    ("Decision Tree", dt_accuracy, dt_auc, dt_f1, dt_precision, dt_time),
    ("Random Forest", rf_accuracy, rf_auc, rf_f1, rf_precision, rf_time),
    ("GBT Classifier", gbt_accuracy, gbt_auc, gbt_f1, gbt_precision, gbt_time)
]

# Define column names for the summary table
columns = ["Model", "Accuracy", "AUC", "F1 Score", "Precision", "Training Time"]

# Convert the data into a pandas DataFrame and display the table
summary_df = pd.DataFrame(data, columns=columns)
summary_df

,Model,Accuracy,AUC,F1 Score,Precision,Training Time
0,Logistic Regression,0.635175,0.679042,0.629832,0.635862,481.332653
1,Decision Tree,0.704703,0.669855,0.704703,0.704703,306.173916
2,Random Forest,0.697418,0.769555,0.696473,0.697084,1905.044151
3,GBT Classifier,0.718196,0.795335,0.718085,0.718026,1757.120425


Best Model Selection

In [21]:
# The best performing model is selected based on AUC and F1 Score, as they are more reliable for imbalanced datasets like HIGGS dataset.
print("The best model for evaluation is: GBT Classifier")

The best model for evaluation is: GBT Classifier


Task 3 Conclusion

In [22]:
# 4 machine learning models were implemented and evaluated on the HIGGS dataset: 
# Logistic Regression, Decision Tree, Random Forest, and Gradient Boosted Trees. 
# Among all models, GBT performed better than other individual models 
# due to its ability to reduce overfitting and capture complex patterns. 
# Overall, GBT achieved the best performance in terms of AUC and F1 Score.